# UV Index Conditions and Public Warning Thresholds

**Project title:** Communicating Useful Weather Indicators for Public Weather Guidance in Malaysia  
**Group members:** LYM, Cedric, Xana, Chen, Xu Zihao  
**Individual focus:** Xu Zihao  

## Research question

**Which weather conditions correspond to the highest UV index values, and at what threshold should public UV warnings be issued?**

This notebook contributes the UV-risk part of the group project. The client is a Malaysian public weather information service that needs weather indicators that are easy for residents, commuters, and outdoor users to understand. The goal of this section is to identify when UV risk becomes high and how it should be communicated as a public-facing warning.

## 1. Client context and variables

The UV Index is the headline variable for this analysis because it directly measures the intensity of ultraviolet radiation. Weather variables such as temperature, humidity, rain and wind are used as supporting context to describe the conditions that tend to appear when the UV Index is high.

| Variable | Unit / meaning | Role in this analysis |
|---|---|---|
| `uv_index` | UV radiation index, normally reported from 0 to 11+ | Main risk indicator |
| `temperature` | Degrees Celsius | Checks whether high UV tends to occur in hotter conditions |
| `humidity` | Percentage | Checks whether high UV is linked with drier or more humid conditions |
| `dew_point` | Degrees Celsius | Supporting moisture indicator |
| `precipitation_rate` | Rainfall intensity | Checks whether high UV mainly occurs during dry conditions |
| `wind_speed` and `gust` | Wind conditions | Supporting outdoor condition indicators |
| `hour`, `month`, `state`, `city` | Time and location information | Identifies when and where high UV appears |

The analysis does **not** treat temperature or humidity as replacements for UV Index. They are used only to explain the weather context around high UV values.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd()

candidate_paths = [
    ROOT / "data" / "malaysia_weather_cleaned.csv",
    ROOT / "malaysia_weather_cleaned.csv",
    ROOT / "malaysia_weather_cleaned(3).csv",
    ROOT / "malaysia_weather_cleaned(1).csv",
    ROOT / "malaysia_weather_data.csv",
]

expected_columns = {"uv_index", "temperature", "humidity", "pressure", "wind_speed", "precipitation_rate"}


def read_weather_csv(path: Path) -> pd.DataFrame:
    """Read the CSV in a way that works for both normal CSV files and files with one extra title row."""
    for skip in [0, 1]:
        try:
            df_try = pd.read_csv(path, skiprows=skip, low_memory=False)
        except Exception:
            continue
        if expected_columns.issubset(set(df_try.columns)):
            return df_try
    raise ValueError(f"The file was found, but it does not contain the expected weather columns: {path}")


data_path = None
for path in candidate_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find a weather CSV file. Place malaysia_weather_cleaned.csv in the data folder "
        "or update candidate_paths in this cell."
    )

weather_raw = read_weather_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print("Shape:", weather_raw.shape)
display(weather_raw.head())

## 2. Data cleaning and preparation

The dataset contains some missing values, especially in UV and precipitation fields. For this research question, missing UV Index values are **not** changed to zero because zero and missing mean different things:

- `uv_index = 0` means the recorded UV level is zero or near zero.
- missing `uv_index` means the UV value was not available in the dataset.

Only rows with a valid UV Index are used for the main UV analysis. Supporting weather variables are converted to numeric values so that summaries, categories and plots are reliable.

In [ ]:
weather = weather_raw.copy()

# Convert key numeric columns safely.
numeric_columns = [
    "temperature", "humidity", "dew_point", "pressure", "wind_speed", "gust",
    "uv_index", "precipitation_rate", "precipitation_total", "year", "month_num",
    "day", "hour", "minutes", "seconds"
]

for col in numeric_columns:
    if col in weather.columns:
        weather[col] = pd.to_numeric(weather[col], errors="coerce")

# Create a usable datetime column. The cleaned file already has datetime; the raw file can be reconstructed.
if "datetime" in weather.columns:
    weather["datetime"] = pd.to_datetime(weather["datetime"], errors="coerce")
elif {"year", "month", "day", "hour", "minutes", "seconds"}.issubset(weather.columns):
    month_map = {
        "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
        "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
    }
    weather["month_num"] = weather["month"].map(month_map)
    weather["datetime"] = pd.to_datetime(
        {
            "year": weather["year"],
            "month": weather["month_num"],
            "day": weather["day"],
            "hour": weather["hour"],
            "minute": weather["minutes"],
            "second": weather["seconds"],
        },
        errors="coerce"
    )

# Keep only rows with a valid UV Index for this section.
uv_df = weather.dropna(subset=["uv_index"]).copy()

# Make simple client-facing categories.
uv_bins = [-0.1, 2, 5, 7, 10, np.inf]
uv_labels = ["Low (0-2)", "Moderate (3-5)", "High (6-7)", "Very high (8-10)", "Extreme (11+)"
]
uv_df["uv_category"] = pd.cut(uv_df["uv_index"], bins=uv_bins, labels=uv_labels)

uv_df["daylight_uv"] = uv_df["uv_index"] > 0
uv_df["sun_protection_needed"] = uv_df["uv_index"] >= 3
uv_df["public_warning"] = uv_df["uv_index"] >= 6
uv_df["very_high_alert"] = uv_df["uv_index"] >= 8
uv_df["extreme_alert"] = uv_df["uv_index"] >= 11
uv_df["wet_weather"] = uv_df["precipitation_rate"].fillna(0) > 0

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Rows loaded",
        "Rows with valid UV Index",
        "Rows with missing UV Index",
        "Rows where UV Index = 0",
        "Rows where UV Index > 0",
        "Rows where UV Index >= 6",
        "Rows where UV Index >= 8",
        "Maximum UV Index",
        "Earliest timestamp",
        "Latest timestamp",
        "States represented"
    ],
    "Value": [
        len(weather),
        len(uv_df),
        int(weather["uv_index"].isna().sum()),
        int((uv_df["uv_index"] == 0).sum()),
        int(uv_df["daylight_uv"].sum()),
        int(uv_df["public_warning"].sum()),
        int(uv_df["very_high_alert"].sum()),
        uv_df["uv_index"].max(),
        uv_df["datetime"].min(),
        uv_df["datetime"].max(),
        uv_df["state"].nunique() if "state" in uv_df.columns else np.nan
    ]
})

display(cleaning_summary)

### Cleaning conclusion

The UV analysis uses only observations where `uv_index` is available. This keeps the interpretation clear because missing UV values should not be treated as zero-risk readings. The dataset is suitable for identifying UV warning thresholds because it includes the full range from low UV values to extreme UV values.

## 3. UV Index distribution

The first step is to check how often each UV risk category appears in the valid UV records. This gives the client a simple view of how common low, moderate, high, very high and extreme UV conditions are in the dataset.

In [ ]:
category_summary = (
    uv_df.groupby("uv_category", observed=False)
    .agg(records=("uv_index", "size"), average_uv=("uv_index", "mean"), median_uv=("uv_index", "median"))
    .reset_index()
)

# For the daylight share, count only records with UV Index > 0.
daylight_counts = (
    uv_df[uv_df["daylight_uv"]]
    .groupby("uv_category", observed=False)
    .size()
    .reindex(category_summary["uv_category"])
    .fillna(0)
    .astype(int)
    .to_numpy()
)

category_summary["daylight_records"] = daylight_counts
category_summary["share_of_valid_uv_records"] = category_summary["records"] / len(uv_df)
category_summary["share_of_daylight_uv_records"] = category_summary["daylight_records"] / uv_df["daylight_uv"].sum()

category_display = category_summary.copy()
category_display["share_of_valid_uv_records"] = category_display["share_of_valid_uv_records"].map(lambda x: f"{x:.1%}")
category_display["share_of_daylight_uv_records"] = category_display["share_of_daylight_uv_records"].map(lambda x: f"{x:.1%}")
display(category_display)

plt.figure(figsize=(9, 5))
plt.bar(category_summary["uv_category"].astype(str), category_summary["records"])
plt.title("Number of observations by UV Index category")
plt.xlabel("UV Index category")
plt.ylabel("Number of records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Distribution conclusion

Most valid UV records are in the low category, mainly because many observations occur outside the strongest sunlight period. However, the dataset still contains many high-risk readings. This means the client should not communicate UV only as a rare extreme event. UV should be shown routinely during the day, with stronger alerts when it reaches high-risk levels.

## 4. Time conditions: when high UV occurs

UV risk is expected to be most important around the middle of the day. The next analysis checks the average UV Index and warning rate by hour.

In [ ]:
hour_summary = (
    uv_df.groupby("hour")
    .agg(
        records=("uv_index", "size"),
        mean_uv=("uv_index", "mean"),
        median_uv=("uv_index", "median"),
        max_uv=("uv_index", "max"),
        warning_rate_ge6=("public_warning", "mean"),
        very_high_rate_ge8=("very_high_alert", "mean")
    )
    .reset_index()
)

hour_display = hour_summary.copy()
hour_display["warning_rate_ge6"] = hour_display["warning_rate_ge6"].map(lambda x: f"{x:.1%}")
hour_display["very_high_rate_ge8"] = hour_display["very_high_rate_ge8"].map(lambda x: f"{x:.1%}")
display(hour_display)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(data=hour_summary, x="hour", y="mean_uv", marker="o", ax=axes[0])
axes[0].axhline(3, linestyle="--", label="Sun protection: UV >= 3")
axes[0].axhline(6, linestyle="--", label="Warning: UV >= 6")
axes[0].set_title("Average UV Index by hour")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Average UV Index")
axes[0].legend()

sns.lineplot(data=hour_summary, x="hour", y="warning_rate_ge6", marker="o", ax=axes[1])
axes[1].set_title("Share of records with UV Index >= 6 by hour")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Warning rate")
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.tight_layout()
plt.show()

In [ ]:
# A compact table showing the main late-morning to mid-afternoon warning window.
main_warning_window = hour_summary[hour_summary["hour"].between(10, 16)].copy()
main_warning_window["warning_rate_ge6"] = main_warning_window["warning_rate_ge6"].map(lambda x: f"{x:.1%}")
main_warning_window["very_high_rate_ge8"] = main_warning_window["very_high_rate_ge8"].map(lambda x: f"{x:.1%}")
display(main_warning_window[["hour", "records", "mean_uv", "median_uv", "max_uv", "warning_rate_ge6", "very_high_rate_ge8"]])

### Time conclusion

High UV values are concentrated around late morning to mid-afternoon, especially from about 10:00 to 16:00. The highest average UV levels occur around midday and early afternoon. For the client, this means UV warnings should be time-aware rather than treated as a whole-day message with the same urgency at every hour.

## 5. Weather conditions linked with high UV

The research question asks which weather conditions correspond to the highest UV Index values. To answer this, the next table compares weather variables across UV risk categories.

In [ ]:
condition_summary = (
    uv_df.groupby("uv_category", observed=False)
    .agg(
        records=("uv_index", "size"),
        median_uv=("uv_index", "median"),
        mean_temperature=("temperature", "mean"),
        median_temperature=("temperature", "median"),
        mean_humidity=("humidity", "mean"),
        median_humidity=("humidity", "median"),
        mean_dew_point=("dew_point", "mean"),
        median_pressure=("pressure", "median"),
        median_wind_speed=("wind_speed", "median"),
        wet_weather_rate=("wet_weather", "mean"),
        median_hour=("hour", "median")
    )
    .reset_index()
)

condition_display = condition_summary.copy()
condition_display["wet_weather_rate"] = condition_display["wet_weather_rate"].map(lambda x: f"{x:.1%}")
display(condition_display)

In [ ]:
# Correlation gives a quick numerical check. It shows association, not causation.
corr_vars = [
    "uv_index", "temperature", "humidity", "dew_point", "pressure", "wind_speed", "gust",
    "precipitation_rate", "precipitation_total", "hour", "month_num"
]
corr_vars = [col for col in corr_vars if col in uv_df.columns]
correlation_with_uv = (
    uv_df[corr_vars]
    .corr(numeric_only=True)["uv_index"]
    .drop("uv_index")
    .sort_values()
)

display(correlation_with_uv.to_frame("correlation_with_uv_index"))

plt.figure(figsize=(9, 5))
correlation_with_uv.plot(kind="barh")
plt.title("Correlation between UV Index and weather variables")
plt.xlabel("Correlation with UV Index")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

In [ ]:
# Temperature-humidity combinations help translate the pattern into public-friendly weather conditions.
uv_df["temperature_bin"] = pd.cut(
    uv_df["temperature"],
    bins=[-np.inf, 25, 28, 31, 34, np.inf],
    labels=["<=25", "25-28", "28-31", "31-34", ">34"]
)
uv_df["humidity_bin"] = pd.cut(
    uv_df["humidity"],
    bins=[-np.inf, 60, 70, 80, 90, np.inf],
    labels=["<=60", "60-70", "70-80", "80-90", ">90"]
)

risk_grid = (
    uv_df.dropna(subset=["temperature_bin", "humidity_bin"])
    .groupby(["humidity_bin", "temperature_bin"], observed=False)["public_warning"]
    .mean()
    .unstack()
)

plt.figure(figsize=(9, 5))
sns.heatmap(risk_grid, annot=True, fmt=".0%", cmap="YlOrRd", vmin=0, vmax=1)
plt.title("Probability of public UV warning (UV Index >= 6) by temperature and humidity")
plt.xlabel("Temperature band (°C)")
plt.ylabel("Humidity band (%)")
plt.tight_layout()
plt.show()

risk_grid_display = risk_grid.copy()
for col in risk_grid_display.columns:
    risk_grid_display[col] = risk_grid_display[col].map(lambda x: f"{x:.1%}" if pd.notna(x) else "")
display(risk_grid_display)

### Weather-condition conclusion

The highest UV Index values in this dataset mainly correspond to:

1. **Warmer conditions**: high and very high UV categories have median temperatures around the low 30s °C.
2. **Lower humidity than low-UV periods**: high UV is more common when humidity is around the mid-60% range rather than above 80% or 90%.
3. **Dry or non-raining observations**: current rain is rare when UV is high, so rainfall should not be used to explain high UV risk.
4. **Midday and early-afternoon timing**: time of day is one of the clearest practical cues for communication.

This does not mean temperature or humidity cause UV Index by themselves. The safer interpretation is that high UV tends to occur during bright daytime conditions that are also warmer and less humid in this dataset.

## 6. Selecting a public UV warning threshold

International UV Index categories commonly define:

- **0-2** as low
- **3-5** as moderate
- **6-7** as high
- **8-10** as very high
- **11+** as extreme

For public communication, there is a difference between a general sun-protection reminder and a stronger warning. A reminder can start at **UV Index >= 3**, while a stronger public warning should start at **UV Index >= 6**, because this is the point where the UV category becomes **high**.

In [ ]:
threshold_rows = []
for threshold, label, client_action in [
    (3, "Moderate or above", "Show sun-protection reminder"),
    (6, "High or above", "Issue public UV warning"),
    (8, "Very high or above", "Escalate warning / extra protection"),
    (11, "Extreme", "Urgent extreme UV alert"),
]:
    mask = uv_df["uv_index"] >= threshold
    threshold_rows.append({
        "threshold": f"UV Index >= {threshold}",
        "category_level": label,
        "records": int(mask.sum()),
        "share_of_valid_uv_records": mask.mean(),
        "share_of_daylight_uv_records": mask.sum() / uv_df["daylight_uv"].sum(),
        "share_occurring_between_10_and_16": uv_df.loc[mask, "hour"].between(10, 16).mean(),
        "recommended_client_action": client_action
    })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_display = threshold_summary.copy()
for col in ["share_of_valid_uv_records", "share_of_daylight_uv_records", "share_occurring_between_10_and_16"]:
    threshold_display[col] = threshold_display[col].map(lambda x: f"{x:.1%}")

display(threshold_display)

### Threshold conclusion

The recommended threshold for a **public UV warning** is:

## **UV Index >= 6**

This threshold is strong enough to avoid over-warning the public for every moderate UV reading, but early enough to warn before the UV level reaches very high or extreme categories. The dataset supports this threshold because UV Index values of 6 or above are concentrated in the late morning to mid-afternoon period, making the warning practical and easy to communicate.

Recommended public-facing wording:

> **UV Warning:** UV Index is 6 or above. Avoid long unprotected outdoor exposure between late morning and mid-afternoon. Use shade, sunscreen, protective clothing, a hat and sunglasses.

A second, stronger message should be used when the UV Index reaches **8 or above**:

> **Very High UV Alert:** UV Index is 8 or above. Extra protection is needed and outdoor exposure should be reduced where possible.

## 7. Final answer to the research question

**Which weather conditions correspond to the highest UV Index values?**

The highest UV Index values are most strongly associated with daytime conditions, especially late morning to mid-afternoon. They also tend to occur when temperature is higher, humidity is lower, and there is little or no current rainfall. Temperature and humidity are useful supporting descriptions, but they should not replace the UV Index itself.

**At what threshold should public UV warnings be issued?**

The client should issue a public UV warning at **UV Index >= 6**. This matches the beginning of the high-risk category and gives the public a clear action point. A lighter sun-protection reminder can begin at **UV Index >= 3**, while stronger alerts should be used at **UV Index >= 8** and especially at **UV Index >= 11**.

### Contribution to the group conclusion

For the final group recommendation, UV Index should be treated as an **always-show daytime indicator** and a **conditional alert indicator**:

- Always display daytime UV Index when available.
- Use **UV >= 3** for a general sun-protection reminder.
- Use **UV >= 6** for a public UV warning.
- Use **UV >= 8** for a stronger very-high-risk alert.
- Do not replace UV Index with temperature or humidity, because those variables only describe context and do not directly measure UV radiation.

## 8. Limitations

- The dataset covers available observations in the provided file, not every place and time in Malaysia.
- Some UV Index values are missing. These missing values are not treated as zero, so the results depend on records where UV was available.
- The analysis is descriptive. It identifies conditions associated with high UV, but it does not prove that temperature, humidity or wind cause UV to rise.
- Cloud cover is not available in the dataset. This is important because cloud conditions can affect UV radiation reaching the ground.
- A few high UV readings appear outside the expected main daylight window. This may reflect timestamp or observation-timing issues, so a real public warning system should verify time-zone handling before deployment.
- A stronger forecasting model would need more complete time-series data and additional variables such as cloud cover, solar radiation and air quality.

## 9. References

- World Health Organization (2022) *Radiation: The ultraviolet (UV) index*. Available at: https://www.who.int/news-room/questions-and-answers/item/radiation-the-ultraviolet-%28uv%29-index
- World Health Organization (2022) *Ultraviolet radiation*. Available at: https://www.who.int/news-room/fact-sheets/detail/ultraviolet-radiation
- United States Environmental Protection Agency (n.d.) *UV Index Scale*. Available at: https://www.epa.gov/sunsafety/uv-index-scale-0
- National Environment Agency Singapore (n.d.) *Ultraviolet Index*. Available at: https://www.nea.gov.sg/corporate-functions/weather/ultraviolet-index